# LSE market intelligence tour

A walk through London Strategic Edge's extended datasets beyond plain candles, tying
each one back into `maroczy`'s own research tooling:

1. **Options flow** -- unusual large-premium prints, ranked into a "smart money" leaderboard.
2. **Macro dashboard** -- the 10Y-2Y yield curve as a classic recession signal.
3. **Event study** -- realized volatility around CPI prints.
4. **Insider-trading signal** -- do open-market insider purchases predict forward returns?
5. **Multi-asset bubble screener** -- Phillips, Shi & Yu (2015) GSADF run across a basket
   pulled straight from the LSE catalog.

Requires `pip install "maroczy[lse,notebook]"` and an `LSE_API_KEY` (see `.env.example`).

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from maroczy.datafeed import LSEData
from maroczy.features import bubble

lse = LSEData()


def find_column(df: pd.DataFrame, candidates: list[str], required: bool = True) -> str | None:
    """Resolve the first matching column name; LSE's exact field names for some of
    the reference/event endpoints below aren't pinned down in the public docs, so we
    check common aliases rather than hardcoding a guess."""
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found in columns: {df.columns.tolist()}")
    return None

## 1. Options flow: unusual activity scanner

Sweep the whole options tape for large premium prints and rank underlyings by net
call-minus-put premium.

In [ ]:
flow = lse.options_flow(min_premium=250_000, max_dte=45)
print(f"{len(flow)} large prints (>$250k premium, <=45 DTE)")
flow.head()

In [ ]:
COL_UNDERLYING = find_column(flow, ["underlying", "symbol", "ticker"])
COL_PREMIUM = find_column(flow, ["premium", "total_premium", "notional"])
COL_FLOW_TYPE = find_column(flow, ["type", "option_type", "right"])

flow["signed_premium"] = np.where(
    flow[COL_FLOW_TYPE].astype(str).str.lower().str.startswith("c"),
    flow[COL_PREMIUM],
    -flow[COL_PREMIUM],
)
leaderboard = (
    flow.groupby(COL_UNDERLYING)["signed_premium"]
    .sum()
    .sort_values(key=np.abs, ascending=False)
    .head(20)
    .rename("net_call_minus_put_premium")
)
leaderboard

In [ ]:
fig_flow = px.bar(
    leaderboard.sort_values(),
    orientation="h",
    title="Options flow leaderboard: net call - put premium (>$250k prints, <=45 DTE)",
    labels={"value": "net premium ($)", "index": "underlying"},
)
fig_flow.show()

## 2. Macro dashboard: the yield curve as a recession signal

The 10Y-2Y Treasury spread is a classic recession indicator: inversions (spread < 0)
have historically preceded US recessions.

In [ ]:
y10 = lse.bond_yields("US10Y", start="1990-01-01")
y2 = lse.bond_yields("US2Y", start="1990-01-01")

y10_col = find_column(y10, ["value", "yield", "close"])
y2_col = find_column(y2, ["value", "yield", "close"])
date_col_y10 = find_column(y10, ["date", "ts"])
date_col_y2 = find_column(y2, ["date", "ts"])

curve = pd.merge(
    y10[[date_col_y10, y10_col]].rename(columns={date_col_y10: "date", y10_col: "us10y"}),
    y2[[date_col_y2, y2_col]].rename(columns={date_col_y2: "date", y2_col: "us2y"}),
    on="date",
)
curve["date"] = pd.to_datetime(curve["date"])
curve["spread_10y_2y"] = curve["us10y"] - curve["us2y"]

fig_curve = go.Figure()
fig_curve.add_trace(go.Scatter(x=curve["date"], y=curve["spread_10y_2y"], name="10Y-2Y spread"))
fig_curve.add_hline(y=0, line_dash="dash", line_color="red")
fig_curve.update_layout(
    title="US 10Y-2Y Treasury spread (negative = inverted curve)",
    yaxis_title="spread (percentage points)",
)
fig_curve.show()

## 3. Event study: realized volatility around CPI prints

Does realized volatility spike around inflation prints? Compare SPY's annualized
realized vol in a window around each released CPI date against the full-sample baseline.

In [ ]:
calendar = lse.economic_calendar(region="US", event="cpi", released_only=True)
date_col_cal = find_column(calendar, ["datetime", "date"])
calendar[date_col_cal] = pd.to_datetime(calendar[date_col_cal])

spy = lse.candles("SPY", "1d", start="2010-01-01")
spy_ret = np.log(spy["close"]).diff().dropna()

WINDOW_DAYS = 3
rows = []
for event_date in calendar[date_col_cal].dt.normalize().unique():
    window = spy_ret.loc[event_date - pd.Timedelta(days=WINDOW_DAYS) : event_date + pd.Timedelta(days=WINDOW_DAYS)]
    if len(window) > 2:
        rows.append({"event_date": event_date, "realized_vol": window.std() * np.sqrt(252)})

event_vol = pd.DataFrame(rows).set_index("event_date")
baseline_vol = spy_ret.std() * np.sqrt(252)
print(f"baseline annualized vol:                 {baseline_vol:.2%}")
print(f"avg annualized vol in +/-{WINDOW_DAYS}d CPI windows: {event_vol['realized_vol'].mean():.2%}")
event_vol.tail()

## 4. Insider-purchase signal: does it predict forward returns?

Pull open-market insider purchases (SEC form code `P-Purchase`) across the most-active
names and measure the average forward return over a fixed holding period after each
purchase -- a classic "follow the insiders" factor idea, quickly sanity-checked.

In [ ]:
insiders = lse.insider_trades(type="P-Purchase", start="2023-01-01")
COL_SYM = find_column(insiders, ["symbol", "ticker"])
COL_TXN_DATE = find_column(insiders, ["transaction_date", "date"])
insiders[COL_TXN_DATE] = pd.to_datetime(insiders[COL_TXN_DATE])

top_symbols = insiders[COL_SYM].value_counts().head(15).index.tolist()
print(f"Scoring the insider-purchase signal across {len(top_symbols)} most-active symbols:")
print(top_symbols)

In [ ]:
HOLD_DAYS = 21
forward_returns = []
for symbol in top_symbols:
    try:
        bars = lse.candles(symbol, "1d", start="2022-06-01")
    except Exception as exc:
        print(f"skip {symbol}: {exc}")
        continue
    close = bars["close"]
    events = insiders.loc[insiders[COL_SYM] == symbol, COL_TXN_DATE]
    for event_date in events:
        after = close.loc[close.index >= event_date]
        if len(after) > HOLD_DAYS:
            fwd_ret = after.iloc[HOLD_DAYS] / after.iloc[0] - 1
            forward_returns.append({"symbol": symbol, "event_date": event_date, "fwd_return": fwd_ret})

signal_df = pd.DataFrame(forward_returns)
print(f"{len(signal_df)} insider-purchase events with a {HOLD_DAYS}-trading-day forward return")
if not signal_df.empty:
    print(f"mean forward return: {signal_df['fwd_return'].mean():.2%}   hit rate: {(signal_df['fwd_return'] > 0).mean():.1%}")
signal_df.head()

## 5. Multi-asset bubble screener (Phillips, Shi & Yu, 2015)

Run GSADF across a basket pulled straight from the LSE catalog to flag which assets
currently show statistically significant explosive ("bubble") behavior.

In [ ]:
catalog = lse.catalog("crypto")
COL_CAT_SYM = find_column(catalog, ["symbol"])
if "ticks" in catalog.columns:
    universe = catalog.sort_values("ticks", ascending=False)[COL_CAT_SYM].head(10).tolist()
else:
    universe = catalog[COL_CAT_SYM].head(10).tolist()
print("screening:", universe)

In [ ]:
results = []
for symbol in universe:
    try:
        bars = lse.candles(symbol, "1d", start="2023-01-01")
    except Exception as exc:
        print(f"skip {symbol}: {exc}")
        continue
    if len(bars) < 100:
        continue
    result = bubble.gsadf(bars["close"])
    cv_95 = bubble.critical_values(len(bars))["gsadf"][0.95]
    results.append({"symbol": symbol, "gsadf_stat": result.stat, "cv_95": cv_95, "excess": result.stat - cv_95})

screen = pd.DataFrame(results).sort_values("excess", ascending=False)
screen

In [ ]:
fig_screen = px.bar(
    screen,
    x="symbol",
    y="excess",
    color="excess",
    color_continuous_scale="RdYlGn_r",
    title="Bubble screen: GSADF stat minus 95% critical value (positive = statistically explosive)",
)
fig_screen.add_hline(y=0, line_dash="dash")
fig_screen.show()

## Caveats

- Column names across LSE's reference/event endpoints are resolved defensively
  (`find_column`) since exact field names weren't confirmed against a live response in
  this environment -- extend the alias lists above if your responses differ.
- The insider-purchase and bubble screens here are illustrative research templates, not
  vetted trading signals. Validate properly with `maroczy.features.cpcv`
  (Combinatorial Purged Cross-Validation) and genuine out-of-sample testing before
  sizing any real capital against them.
- The options-flow premium sign convention (call = +, put = -) is a simplification: it
  doesn't distinguish opening vs. closing trades, or buy-side vs. sell-side prints.
- The CPI event study uses a simple +/- N trading-day window around the release date;
  it does not control for other macro releases landing in the same window.